In [29]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import cortex.freesurfer
import sys
import os
import csv
import scipy.stats as st
import pandas as pd
from scipy.stats import shapiro

''' This is my code to visualize the Population Receptive Field Mapping properties
in two groups, healthy controls and glaucoma patients.'''

' This is my code to visualize the Population Receptive Field Mapping properties\nin two groups, healthy controls and glaucoma patients.'

In [30]:
class PRFModel:
    '''Class representing pRF model parameters '''
    def __init__(self, r2, size, ecc, angle):
        """
        Initialize the PRFModel instance with the following parameters:
            
        Parameters:
        r2 (float): Model fit value (R-squared). Represents how well the model explains the variance in the observed data. 
                    A higher value indicates a better fit.
        size (float): Size of the pRF (population receptive field), which describes the spatial extent or diameter of the receptive field.
        ecc (float): Eccentricity. Represents the distance of the pRF from the center of the visual field. Higher values mean the receptive field 
                    is located farther from the center (fovea).
        angle (float): Polar angle. Specifies the direction of the receptive field from the center, often in degrees or radians.
        """
        self.r2 = r2            # Model fit - represents how well the model explains data variance
        self.size = size        # Size of the pRF (population receptive field)
        self.ecc = ecc          # Eccentricity - distance of the pRF from the center
        self.angle = angle      # Polar angle - position angle of the pRF

def load_pickle_file(filepath):
    '''Function to load a pickle file given a file path'''
    # Check if the given file path points to an existing file
    if not os.path.isfile(filepath):
        # If the file does not exist, raise a FileNotFoundError
        raise FileNotFoundError(f"File not found: {filepath}")
    # Open the file in 'rb' mode (read binary) since pickle files contain serialized binary data
    with open(filepath, 'rb') as file: 
        # Deserializes the contents of the binary file and returns the Python object.
        return pickle.load(file)

def load_prf_params(subjectid, main_path, atlas, denoising):
    '''Function to load pRF parameters for a given subjects and other details'''
    # Construct the file path to the pRF parameters file based on input parameters
    # The path points to a pickle file that contains the pRF model information
    filepath = os.path.join(main_path, f'pRFM/{subjectid}/ses-02/{denoising}/model-{atlas}-nelder-mead-GM_desc-prf_params.pkl') 
    # Load the pickle file using the load_pickle_file function defined previously
    pkl_data = load_pickle_file(filepath)                       
    # Extract the pRF model parameters using the 'iterative_search_params' key from the loaded data
    prf_params = pkl_data['model'].iterative_search_params     
    # Extract the indices of the voxels that fall within the region of interest (ROI) 
    # This is done by finding where the 'rois_mask' is equal to 1 (indicating the voxel is inside the ROI)
    prf_voxels = np.where(pkl_data['rois_mask'] == 1)[0]      
    # Return both the extracted pRF model parameters and the indices of the voxels within the ROI
    return prf_params, prf_voxels

def load_labels(subjectid, main_path, atlas):
    '''Function to load FreeSurfer labels for the specified subject'''
    # Define the path to the FreeSurfer directory within the main project directory
    fs_dir = os.path.join(main_path, 'freesurfer') 
    try:
        # Load label indices for visual areas and eccentricity regions:
        # Load visual area labels (V1, V2, V3, etc.) using Benson's visual area label ('benson14_varea-0001')
        idx_rois4, idx_vls4 = cortex.freesurfer.get_label(subjectid, label='benson14_varea-0001', fs_dir=fs_dir, hemisphere=('lh', 'rh'))
        # Load eccentricity labels using Benson's eccentricity label ('benson14_eccen-0001')
        idx_rois1, idx_vls1 = cortex.freesurfer.get_label(subjectid, label='benson14_eccen-0001', fs_dir=fs_dir, hemisphere=('lh', 'rh'))
        # Check if 'manual' atlas is specified
        # If a manual atlas is requested, load the manual labels and update the corresponding indices
        if atlas == 'manual': 
            idx_rois_manual, idx_vls_manual = cortex.freesurfer.get_label(subjectid, label='manualdelin', fs_dir=fs_dir, hemisphere=('lh', 'rh'))
            # Update the values in idx_vls4 to use the manual labels wherever applicable
            idx_vls4[idx_rois_manual] = idx_vls_manual
    except Exception as e:
        # If there is an error loading any of the labels, raise a RuntimeError
        raise RuntimeError(f"Error loading labels for subject {subjectid}: {e}")
    # Return the loaded indices and values:
    # - idx_rois4, idx_vls4: Visual area indices and labels
    # - idx_vls1: Eccentricity labels
    return idx_rois4, idx_vls4, idx_vls1

def filter_prf_data(prf_params, prf_voxels):
    '''Filter and extract pRF data for the selected voxels'''
    # Construct and return a PRFModel object containing specific parameters of interest
    # - r2, size, eccentricity, and polar angle for the selected voxels.
    return PRFModel(
        r2=prf_params[:, 7],                                                # Extract R2 values from column 7 of prf_params.
        size=prf_params[:, 2],                                              # Extract pRF size values from column 2 of prf_params.
        # Eccentricity is calculated as the Euclidean distance from the center (0,0)
        # This is done using the Pythagorean theorem: ecc = sqrt(x^2 + y^2).
        ecc=np.sqrt(prf_params[:, 1]**2 + prf_params[:, 0]**2),             # Calculate eccentricity 
        # np.arctan2(y, x) computes the angle (in radians) of the vector formed by (x, y) relative to the positive x-axis.
        # The function np.rad2deg() converts the resulting angle to degrees.
        angle=np.rad2deg(np.arctan2(prf_params[:, 1], prf_params[:, 0]))    # Calculate polar angle
    )

def save_trendline_data_to_csv(data, output_path):
    '''Function to save trendline data to CSV without duplicating subjects and visual areas'''
    header = ['Subject Number', 'Visual Area', 'Offset', 'Slope', 'Confidence Interval Lower Bound', 'Confidence Interval Upper Bound']
    # Check if file exists
    file_exists = os.path.isfile(output_path)
    # If the file exists, load it and check if the subject and visual area data already exists
    if file_exists:
        existing_data = pd.read_csv(output_path)
        # Create a set of tuples with (Subject Number, Visual Area) to avoid duplicates
        existing_entries = set(zip(existing_data['Subject Number'], existing_data['Visual Area']))
    else:
        existing_entries = set()  # No entries exist yet

    # Open the CSV file in append mode ('a')
    with open(output_path, mode='a', newline='') as file:
        writer = csv.writer(file)
        # If the file doesn't exist, write the header
        if not file_exists:
            writer.writerow(header)
        # Append new data only if the subject number and visual area are not already in the CSV
        for row in data:
            subject_num, visual_area = row[0], row[1]
            if (subject_num, visual_area) not in existing_entries:  # Only add if the subject and visual area are not already in the file
                writer.writerow(row)

def generate_combined_plot(subject_num, prf_model, idx_vls4, prf_voxels, rois_list, pdf_pages, output_dir):
    '''Generate a combined plot with trendlines of V1, V2, V3 for a subject'''
    fig, ax = plt.subplots(figsize=(12, 8))
    colors = {'V1': 'blue', 'V2': 'orange', 'V3': 'green'}

    for roi_name in rois_list[0]:
        # Find the indices of the current ROI within the rois_list
        roi_idx = np.where(roi_name == rois_list[0, :])[0]
        # Find the vertices that correspond to the current ROI
        roi_vertices = np.array(np.where(idx_vls4 == int(rois_list[1, roi_idx])))[0]
        # Identify which voxels belong to the current ROI based on intersection with prf_voxels
        idx = np.in1d(prf_voxels, roi_vertices)

        # Extract parameters for the current ROI based on the filtered voxel indices
        filtered_params = {'ecc': prf_model.ecc[idx], 'angle': prf_model.angle[idx], 'size': prf_model.size[idx], 'r2': prf_model.r2[idx]}

        # Apply validity criteria to select the relevant data points for the plot
        valid_idx = (filtered_params['r2'] > 0.3) & (filtered_params['ecc'] > 0) & (filtered_params['ecc'] < 7) & (filtered_params['size'] > 0.11)
        ecc = filtered_params['ecc'][valid_idx]
        data = filtered_params['size'][valid_idx]
        r2_values = filtered_params['r2'][valid_idx]

        # Sort the data to fit trend lines properly 
        sorted_idx = np.argsort(ecc)
        ecc, data, r2_values = ecc[sorted_idx], data[sorted_idx], r2_values[sorted_idx]
        # Fit the final model to the entire dataset to extract trendline parameters (slope and offset)
        final_model = np.polyfit(ecc, data, deg=1, w=r2_values)
        slope, offset = final_model
        # Generate evenly spaced x values for fitting trendlines within the specified eccentricity bounds
        x_fit = np.linspace(0, 7, len(ecc))
        y_fit = slope * x_fit + offset

        # Plot the trend line for the ROI
        ax.plot(x_fit, y_fit, ls='solid', linewidth=1.5, alpha=1, color=colors.get(roi_name, 'gray'), 
                label=f'{roi_name} (N = {len(r2_values)}): y = {slope:.4f}x + {offset:.4f}')
        # Scatter plot of the data points
        ax.scatter(ecc, data, alpha=r2_values, marker='o', s=1, color=colors.get(roi_name, 'gray'))

    # Add title, labels, limits, and legend to the plot
    ax.set_title(f'{subject_num}: Combined Trendlines for V1, V2, V3', fontsize=18)
    ax.set_xlabel('Eccentricity', fontsize=16)  # Label for the x-axis
    ax.set_ylabel('pRF Size', fontsize=16)  # Label for the y-axis
    ax.set_ylim([0, 3.5])  # Set y-axis limits to maintain consistent scale
    ax.legend(loc='upper left')  # Position the legend in the upper left
    ax.grid(True)  # Display a grid on the plot
    plt.tight_layout()  # Adjust the padding of the plot for a clean layout

    # Save the figure to a PDF file if a PDF page object is provided
    if pdf_pages:
        pdf_pages.savefig(fig)
    plt.close(fig)  # Close the figure to free up memory

def bootstrap_weighted_avg(x, y, r2, n_samples, confidence, ecc_bound, roi_name, subject_num, group_type=None, output_dir=None):
    '''Function to create a bootstrapped weighted average plot'''
    # Set the Seaborn plotting style to 'white' for a clean background
    sns.set_style('white')
    # Assign color based on the group type or ROI name
    colors = {'V1': 'blue', 'V2': 'orange', 'V3': 'green', 'glaucoma': 'red', 'control': 'blue'}
    # Get the color for the group or ROI; default to 'gray' if group/ROI is not in the dictionary
    group_color = colors.get(group_type, colors.get(roi_name, 'gray'))
    # Scatter plot of the x and y data, with transparency set by the R2 value
    plt.scatter(x, y, alpha=r2, marker='o', s=1, color=group_color)

    # Sort the data to fit trend lines properly 
    sorted_idx = np.argsort(x)
    x, y, r2 = x[sorted_idx], y[sorted_idx], r2[sorted_idx]
    # Generate evenly spaced x values for fitting trendlines within the specified eccentricity bounds
    x_fit = np.linspace(ecc_bound[0], ecc_bound[1], len(x))
    y_fit_samples = np.zeros((n_samples, len(x_fit)))

    # Bootstrapping process: create multiple trend lines based on resampled data
    for i in range(n_samples):
        # Randomly choose indices to create a bootstrap sample
        rnd_idx = np.random.choice(len(x), size=len(x), replace=True)
        # Resample the x, y, and r2 data using the random indices
        res_x, res_y, res_r2 = x[rnd_idx], y[rnd_idx], r2[rnd_idx]
        # Fit a linear model (degree 1 polynomial) to the resampled data, weighted by r2
        model = np.polyfit(res_x, res_y, deg=1, w=res_r2)
        # Calculate the y-values for the fitted line and store in y_fit_samples
        y_fit_samples[i, :] = model[0] * x_fit + model[1]

    # Calculate the mean trend line and the confidence interval bounds from bootstrapped samples
    y_mean_fit = np.mean(y_fit_samples, axis=0)
    lower_bounds = np.percentile(y_fit_samples, 2.5, axis=0) # 2.5th percentile for lower bound
    upper_bounds = np.percentile(y_fit_samples, 97.5, axis=0) # 97.5th percentile for upper bound

    # Fit the final model to the entire dataset to extract trendline parameters (slope and offset)
    final_model = np.polyfit(x, y, deg=1, w=r2)
    slope, offset = final_model

    # Plot the trend line based on the mean fit and add a label to the plot
    plt.plot(x_fit, y_mean_fit, ls='solid', linewidth=1.5, alpha=1, color=group_color, 
         label=f'{roi_name} (N = {len(r2)}): y = {np.polyfit(x, y, deg=1, w=r2)[0]:.4f}x + {np.polyfit(x, y, deg=1, w=r2)[1]:.4f}')
     # Plot the confidence interval as a shaded area around the trend line
    plt.fill_between(x_fit, lower_bounds, upper_bounds, color='gray', alpha=0.2)
    # Add title, labels, limits, and legend to the plot
    plt.title(f'{subject_num}: Trendline for {roi_name}', fontsize=18)
    plt.xlabel('Eccentricity', fontsize=16) # Label for the x-axis
    plt.ylabel('pRF Size', fontsize=16) # Label for the y-axis
    plt.ylim([0, 3.5]) # Set y-axis limits to maintain consistent scale
    plt.legend(loc='upper left') # Position the legend in the upper left
    plt.grid(True) # Display a grid on the plot
    plt.tight_layout() # Adjust the padding of the plot for a clean layout

    # Save trendline data to a CSV file
    csv_output_path = os.path.join(output_dir, 'trendline_data.csv') # Define the output CSV file path
    # Prepare the data to be saved in the CSV, including subject, ROI, offset, slope, and confidence bounds
    trendline_data = [[subject_num, roi_name, offset, slope, lower_bounds[0], upper_bounds[0]]]
    save_trendline_data_to_csv(trendline_data, csv_output_path)

def basic_plot(subject_num, ecc, data, r2_values, prop, roi_name, ax):
    '''Basic scatter plot of pRF size versus eccentricity for each ROI '''
    # Scale the transparency of points based on R2 values
    # Normalizes R2 values between 0 and 1, and then scales them between 0.1 and 1.0 for alpha
    alpha = (r2_values - np.min(r2_values)) / (np.max(r2_values) - np.min(r2_values))
    alpha = 0.1 + 0.9 * alpha # Ensures alpha is not too transparent (minimum of 0.1)

    # Plot data points using the scatter plot
    colors = {'V1': 'blue', 'V2': 'orange', 'V3': 'green'}  # Define colors for each ROI
    # If the ROI name is not in the color dictionary, use 'gray' by default
    ax.scatter(ecc, data, alpha=alpha, s=1, color=colors.get(roi_name, 'gray'))

    # Fit a linear trend line to the data
    model = np.polyfit(ecc, data, deg=1) # Fit a degree 1 polynomial (i.e., a line)
    predict = np.poly1d(model) # Create a polynomial function for prediction
    # Plot the fitted trend line
    ax.plot(ecc, predict(ecc), color=colors.get(roi_name, 'red'), label=f'Trend Line ({roi_name} - N = {len(r2_values)})')
    
    # Set the title of the plot, including the subject number and ROI name
    ax.set_title(f'{subject_num}: r2-based Trendline for {roi_name}', fontsize=18)
    # Set the limits for the x-axis (eccentricity range) and y-axis (depends on the property)
    ax.set_xlim([0, 7])
    ax.set_ylim([0, 4] if prop != 'r2' else [0, 1]) # Different limit if plotting 'r2' property
    # Set tick values for the x and y axes to provide better readability
    ax.set_xticks(np.arange(1, 8, 1)) # Set x-axis ticks from 1 to 7, with steps of 1
    ax.set_yticks(np.arange(0, 4.5, 0.5))  # Set y-axis ticks from 0 to 4.5, with steps of 0.5
    # Set the labels for the x and y axes, ensuring proper capitalization for the property label
    ax.set_xlabel('Eccentricity', fontsize=16)
    ax.set_ylabel(prop.capitalize(), fontsize=16)
    ax.legend() # Add a legend to the plot
    ax.grid(True) # Add a grid to the plot for better visual reference
    plt.ylim([0, 3.5]) # Set y-axis limit again to make sure it doesn't exceed a consistent boundary
    plt.tight_layout() # Adjust layout to make sure everything fits well within the plot space

def plot_rois(subject_num, rois_list, idx_vls4, prf_voxels, prf_model, prop, pdf_pages, line_type='trend', specific_roi=None, n_samples=1000, confidence=0.95, basic=False, output_dir=None):
    '''Plot different ROIs for a subject'''
    fig, ax = plt.subplots(figsize=(12, 8)) # Create a new figure and axes for plotting
    # Determine which ROIs to plot; if specific ROI is provided, plot it, otherwise plot the first one in the list
    rois_to_plot = [specific_roi] if specific_roi else rois_list[0] 

    # Loop through each ROI to be plotted
    for roi_name in rois_to_plot:
        # Find the indices of the current ROI within the rois_list
        roi_idx = np.where(roi_name == rois_list[0, :])[0]
        # Find the vertices that correspond to the current ROI
        roi_vertices = np.array(np.where(idx_vls4 == int(rois_list[1, roi_idx])))[0]
        # Identify which voxels belong to the current ROI based on intersection with prf_voxels
        idx = np.in1d(prf_voxels, roi_vertices)

        # Extract parameters for the current ROI based on the filtered voxel indices
        filtered_params = {'ecc': prf_model.ecc[idx], 'angle': prf_model.angle[idx], 'size': prf_model.size[idx], 'r2': prf_model.r2[idx]}

        # Apply validity criteria to select the relevant data points for the plot
        # - R2 value > 0.3 (to ensure good fit quality)
        # - Eccentricity between 0 and 7 (valid range)
        # - Size greater than 0.11 (minimum pRF size)
        valid_idx = (filtered_params['r2'] > 0.3) & (filtered_params['ecc'] > 0) & (filtered_params['ecc'] < 7) & (filtered_params['size'] > 0.11)
        # Extract the filtered eccentricity, property data, and R2 values for plotting
        ecc = filtered_params['ecc'][valid_idx]
        data = filtered_params[prop][valid_idx]
        r2_values = filtered_params['r2'][valid_idx]

        # Plotting: either create a basic scatter plot or perform bootstrapped weighted average
        if basic:
            # Use basic scatter plot with a linear trendline
            basic_plot(subject_num, ecc, data, r2_values, prop, roi_name, ax)
            ax.legend(loc='upper left') # Add legend to the plot
        else:
            # Use bootstrapping to create a weighted average trendline
            bootstrap_weighted_avg(ecc, data, r2_values, n_samples, confidence, ecc_bound=(0, max(ecc)), roi_name=roi_name, subject_num=subject_num, output_dir=output_dir)
    
    # Save the figure to a PDF file if a PDF page object is provided
    if pdf_pages:
        pdf_pages.savefig(fig)
    plt.close(fig)  # Close the figure to free up memory

def generate_plots(subjectid, main_path, atlas, denoising, project, subject_num, pdf_pages=None, line_type='trend', basic=False, output_dir=None):
    '''Generate plots for pRF model parameters'''
    # Define the list of Regions of Interest (ROIs) to be plotted
    # The first row contains the ROI names, and the second row contains corresponding numeric labels
    rois_list = np.array([['V1', 'V2', 'V3'], [1, 2, 3]])
    try:
        # Load pRF model parameters and voxel indices
        prf_params, prf_voxels = load_prf_params(subjectid, main_path, atlas, denoising)
        # Load FreeSurfer labels to identify visual areas in the brain
        idx_rois4, idx_vls4, idx_vls1 = load_labels(subjectid, main_path, atlas)
        # Filter pRF model data for the voxels that belong to relevant visual areas
        prf_model = filter_prf_data(prf_params, prf_voxels)

        # Plot and save figures for each ROI if a PDF object (`pdf_pages`) is provided
        if pdf_pages:
            # Loop through each ROI in the ROI list
            for roi_name in rois_list[0]:
                plot_rois(subject_num, rois_list, idx_vls4, prf_voxels, prf_model, 'size', pdf_pages, line_type=line_type, specific_roi=roi_name, basic=basic, output_dir=output_dir)
        # If basic plotting is requested, plot all ROIs in a single basic plot
        if basic:
            plot_rois(subject_num, rois_list, idx_vls4, prf_voxels, prf_model, 'size', pdf_pages, line_type=line_type, basic=True, output_dir=output_dir)
    
    except Exception as e:
        print(f"An error occurred: {e}")

def histograms_slope(data, output_pdf_path):
    '''Plots histograms of Slope and Offset for each group across all visual areas.'''
    with PdfPages(output_pdf_path) as pdf:
        colors = {'Healthy Controls': 'blue', 'Primary Open-Angle Glaucoma': 'red'}
        visual_areas = ['V1', 'V2', 'V3']
        parameters = ['Slope', 'Offset']

        for param in parameters:
            for group in data['Group'].unique():
                for area in visual_areas:
                    group_data = data[(data['Group'] == group) & (data['Visual Area'] == area)][param]
                    # Extract the slope values for each group for each visual area in trendline_data
                    # Extract the offset values for each group for each visual area in trendline_data
                    # E.I.
                    # 0 (raw for slope V1)    0.142643 (value of slope V1)
                    plt.figure(figsize=(12, 6))
                    sns.histplot(group_data, bins=20, kde=True, color=colors[group], alpha=0.6, edgecolor='black')
                    plt.title(f'{param} - {group} ({area})', fontsize=16)
                    plt.xlabel(param, fontsize=14)
                    plt.ylabel('Frequency', fontsize=14)
                    plt.xticks(fontsize=12)
                    plt.yticks(fontsize=12)
                    plt.grid(axis='y', linestyle='--', alpha=0.7)
                    pdf.savefig()
                    plt.close()

def normality_slope(data, main_path, visual_areas, groups, atlas='manual'):
    '''Performs normality tests for the given data and saves the results to a CSV file.'''
    normality_output_path = os.path.join(main_path, 'trendline_data', f'normality_{atlas}.csv')
    with open(normality_output_path, mode='w', newline='') as normality_file:
        normality_writer = csv.writer(normality_file)
        normality_writer.writerow(['Group', 'Visual Area', 'Parameter', 'p-value'])

        # Loop through each group and visual area to calculate means and perform normality tests
        for group in groups:
            group_data = data[data['Group'] == group] 
            for area in visual_areas:
                # Extract data for the specific group and visual area
                # E.i. 0 (row) sub-02 (subject number) V1 (visual area) 0.703566 (slope) 0.142643 (offset)
                area_data = group_data[group_data['Visual Area'] == area]
                slope = area_data['Slope'] # Extract all the values of the slope for the visual area 
                offset = area_data['Offset'] # Extract all the values of the offset for the visual area 
                # Perform the Shapiro-Wilk test to assess normality of slope and offset
                slope_p = shapiro(slope)[1] # Extract the p value of the shapirot test
                offset_p = shapiro(offset)[1]
                # Write normality test results to CSV file
                normality_writer.writerow([group, area, 'Slope', slope_p])
                normality_writer.writerow([group, area, 'Offset', offset_p])

    histograms_output_path = os.path.join(MAIN_PATH, 'trendline_data', 'output_histograms.pdf')
    histograms_slope(data, histograms_output_path)

def normality_data(subjects, MAIN_PATH, atlas, denoising, alpha=0.05):
    ''' Analyzing pRF Data for Normality Assessment in the Visual Areas of Interest and Generating Plots '''
    pdf_path = f'/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/trendline_data/histonormality_{atlas}.pdf'
    csv_path = f'/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/trendline_data/shapiro_{atlas}.csv'
    
    with open(csv_path, mode='w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile) # Initializes a writer object that will be used to write data to the file.
        csv_writer.writerow(['Subject', 'Parameter', 'Eccentricity Bin', 'ROI', 'p-value', 'Significance']) # Write the header for the CSV file
        with PdfPages(pdf_path) as pdf:
            for subjectid in subjects:
                try:
                    prf_params, prf_params_vx = load_prf_params(subjectid, MAIN_PATH, atlas, denoising) # Load pRF parameters for the specified subject
                    idx_rois4, idx_vls4, idx_vls1 = load_labels(subjectid, MAIN_PATH, atlas) # Load indices for regions of interest and visual labels
                    pRFmodel_instance = filter_prf_data(prf_params, prf_params_vx) # Filter the pRF data based on the loaded parameters
                    roi_list = ['V1', 'V2', 'V3']

                    # Loop over each ROI
                    for roi_name in roi_list:
                        roi_idx = np.where(roi_name == rois_list[0, :]) # Find the index of the ROI
                        roi_verts = np.array(np.where(idx_vls4 == int(rois_list[1, roi_idx])))[0] # Extract vertices corresponding to the ROI
                        idx = np.in1d(prf_params_vx, roi_verts) # Create an index array for the parameter values
                        
                        # Filtering parameters from the pRF model instance
                        filtered_params = {
                            'ecc': pRFmodel_instance.ecc[idx],
                            'angle': pRFmodel_instance.angle[idx],
                            'size': pRFmodel_instance.size[idx],
                            'r2': pRFmodel_instance.r2[idx]}
                        threshold = 0.3 # Setting a threshold for filtering
                        # Creating an index of parameters that meet the filtering criteria
                        min_size = 0.05
                        idx_r2 = filtered_params['r2'] > threshold
                        idx_ecc1 = filtered_params['ecc'] > 0
                        idx_ecc2 = filtered_params['ecc'] < 7
                        idx_size = filtered_params['size'] > min_size
                        idx_threshold = np.where(np.logical_and(np.logical_and(np.logical_and(idx_r2, idx_ecc1), idx_ecc2), idx_size))

                        ecc_bins = np.arange(0, 7) # Return evenly spaced values within 0 and 7
                        pRFsize_per_bin = {ecc_bin: [] for ecc_bin in ecc_bins} # Initialize the variable to store the pRF size per bin
                        polar_angle_per_bin = {ecc_bin: [] for ecc_bin in ecc_bins} # Initialize the variable to store the pRF polar angle per bin
                        r2_per_bin = {ecc_bin: [] for ecc_bin in ecc_bins} # Initialize the variable to store the r2 size per bin

                        # Iterate over each eccentricity bin 
                        for ecc_bin in ecc_bins:
                            # Binary mask created with the following criteria: 
                            # 1) Checks if the eccentricity values at the specified indices are greater than or equal to the lower bound of the bin 
                            # 2) Checks if the eccentricity values are less than the upper bound of the bin 
                            # True if they meet both criteria and False if they do not
                            bin_mask = (filtered_params['ecc'][idx_threshold] >= ecc_bin) & (filtered_params['ecc'][idx_threshold] < ecc_bin + 1)
                            # Assign the True parameters to each bin
                            pRFsize_per_bin[ecc_bin] = filtered_params['size'][idx_threshold][bin_mask] 
                            polar_angle_per_bin[ecc_bin] = filtered_params['angle'][idx_threshold][bin_mask]
                            r2_per_bin[ecc_bin] = filtered_params['r2'][idx_threshold][bin_mask]
                            
                            # Shapiro Wilk Test for normality 
                            # Iterate over the parameters of interests. 
                            for param_name, param_data in zip(['size'], [pRFsize_per_bin[ecc_bin]]):
                                stats, pvalue = shapiro(param_data)
                                if pvalue > 0.05: 
                                    result = 'Normally Distributed'
                                else: 
                                    result = 'Not Normally Distributed'
                                pvalues = [] # Initialize a variable to store the pvalues
                                pvalues.append(pvalue) 
                                csv_writer.writerow([subjectid, param_name, ecc_bin, roi_name, pvalues, result])

                        # Create subplots for histograms 
                        for ecc_bin in ecc_bins:  # Iterate over each eccentricity bin
                            # Check if there are enough data points to create a meaningful histogram
                            if len(pRFsize_per_bin[ecc_bin]) > 0:
                                # Create a new figure for each eccentricity bin
                                fig, ax = plt.subplots(1, 1, figsize=(8, 6))
                                
                                # pRF size plot
                                ax.hist(pRFsize_per_bin[ecc_bin], bins=20, edgecolor='black')
                                ax.set_title(f'Subject {subjectid} - {roi_name}\n pRF Size: Eccentricity Bin {ecc_bin}-{ecc_bin + 1}', fontsize=12)
                                ax.set_xlabel('pRF Size', fontsize=10)
                                ax.set_ylabel('Frequency', fontsize=10)
                                ax.grid(True)

                                # Adjust layout and save the figure to the PDF
                                plt.tight_layout()
                                pdf.savefig(fig)
                                plt.close(fig)
                            else:
                                # If there are not enough data points, skip plotting this bin
                                print(f"Skipping plot for Eccentricity Bin {ecc_bin} due to insufficient data")


                except Exception as e:
                    print(f"Skipping subject {subjectid} due to error: {e}")

def perform_mannwhitney_and_plot_trendlines(data, main_path, visual_areas, groups, atlas='manual'):
    '''Performs Mann-Whitney U tests and plots trendlines for each visual area and group.'''
    # Save Mann-Whitney U test results to a separate CSV file
    mannwhitney_output_path = os.path.join(main_path, 'trendline_data', f'mannwhitney_{atlas}.csv')
    with open(mannwhitney_output_path, mode='w', newline='') as mw_file:
        mw_writer = csv.writer(mw_file)
        mw_writer.writerow(['Visual Area', 'Parameter', 'U-statistic', 'p-value'])
        
        # Perform Mann-Whitney U tests for slope and offset between control and glaucoma groups for each visual area
        for area in visual_areas:
            # For each visual area extract the slope column values for the group number corresponding either to healthy controls or to glaucoma patients
            control_slope = data[(data['Group'] == 'Healthy Controls') & (data['Visual Area'] == area)]['Slope']
            glaucoma_slope = data[(data['Group'] == 'Primary Open-Angle Glaucoma') & (data['Visual Area'] == area)]['Slope']
            # For each visual area extract the offset column values for the group number corresponding either to healthy controls or to glaucoma patients
            control_offset = data[(data['Group'] == 'Healthy Controls') & (data['Visual Area'] == area)]['Offset']
            glaucoma_offset = data[(data['Group'] == 'Primary Open-Angle Glaucoma') & (data['Visual Area'] == area)]['Offset']

            # Comparing with a Mann-Whitney U test the slope of the control group and of the glaucoma group and store the U value and p value
            u_stat_slope, p_value_slope = st.mannwhitneyu(control_slope, glaucoma_slope, alternative='two-sided')
            mw_writer.writerow([area, 'Slope', u_stat_slope, p_value_slope])
            
            # Comparing with a Mann-Whitney U test the offset of the control group and of the glaucoma group and store the U value and p value
            u_stat_offset, p_value_offset = st.mannwhitneyu(control_offset, glaucoma_offset, alternative='two-sided')
            mw_writer.writerow([area, 'Offset', u_stat_offset, p_value_offset])

    # Plotting trend lines for each visual area separately with confidence intervals
    for area in visual_areas:
        plt.figure(figsize=(8, 8))
        for group, color in groups.items():
            group_data = data[data['Group'] == group]
            area_data = group_data[group_data['Visual Area'] == area]
            
            # Calculate the mean slope and offset for the group
            mean_slope = area_data['Slope'].mean()
            mean_offset = area_data['Offset'].mean()

            # Calculate the standard error of the mean (SEM)
            sem_slope = st.sem(area_data['Slope'])
            sem_offset = st.sem(area_data['Offset'])

            # Calculate the 95% confidence intervals using 1.96 (for normal distribution)
            slope_ci_lower = mean_slope - 1.96 * sem_slope
            slope_ci_upper = mean_slope + 1.96 * sem_slope
            offset_ci_lower = mean_offset - 1.96 * sem_offset
            offset_ci_upper = mean_offset + 1.96 * sem_offset

            # Create the x values (eccentricity range)
            x = np.linspace(0, 7, 100)
            # Calculate the trendline using the mean slope and offset
            y = mean_slope * x + mean_offset
            # Count how many unique subjects are included in the current group and visual area.
            n_value = len(area_data['Subject Number'].unique())
            # Plot trend line
            plt.plot(x, y, label=f'{group} (N = {n_value})', color=color, linestyle='-')

            # Plot confidence interval as a shaded region
            y_lower = slope_ci_lower * x + offset_ci_lower
            y_upper = slope_ci_upper * x + offset_ci_upper
            plt.fill_between(x, y_lower, y_upper, color=color, alpha=0.2)

        # Final plot formatting
        plt.ylim(0, 3)
        plt.xlabel('Eccentricity', fontsize=25)
        plt.ylabel('Population Receptive Field Size', fontsize=25)
        plt.legend(loc='upper left', fontsize=18)
        plt.title(f'Visual Area: {area}', fontsize=30)
        plt.grid(True)
        plt.tight_layout()
        plt.show()

In [31]:
# MAIN SCRIPT 
            
if __name__ == "__main__":
    ''' Main block to generate plots and analyze data'''
    plot_type = 'bootstrap'
    MAIN_PATH = '/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives'
    OUTPUT_DIR = os.path.join(MAIN_PATH, 'trendline_data')
    denoising = 'nordic'
    project = 'PROJECT_EGRET-AAA'
    rois_list = np.array([['V1', 'V2', 'V3'], [1, 2, 3]])
    atlas = 'manual'
    analysis = 'normality'  # Change this to 'trendline', 'normality', or 'ttest' depending on the desired analysis

    pdf_path = os.path.join(MAIN_PATH, f'trendline_data/trend_{atlas}_{plot_type}.pdf')
    pdf_pages = PdfPages(pdf_path)
    glaucoma_subjects = ['sub-02', 'sub-04', 'sub-09', 'sub-10', 'sub-11', 'sub-14', 'sub-17']
    controls_subjects = ['sub-22', 'sub-25', 'sub-27', 'sub-28', 'sub-29', 'sub-30', 'sub-31', 'sub-32', 'sub-34', 'sub-35', 'sub-39', 'sub-38']
    allowed_subjects = glaucoma_subjects + controls_subjects
    subjects_dir = os.path.join(MAIN_PATH, 'pRFM')
    subjects = [d for d in os.listdir(subjects_dir) if os.path.isdir(os.path.join(subjects_dir, d)) and d in allowed_subjects]

    if analysis == 'ttest':
        try:
            for subjectid in subjects:
                try:
                    if plot_type == 'bootstrap':
                        generate_plots(subjectid, MAIN_PATH, atlas=atlas, denoising=denoising, project=project, subject_num=subjectid, pdf_pages=pdf_pages, output_dir=OUTPUT_DIR)
                        prf_params, prf_voxels = load_prf_params(subjectid, MAIN_PATH, atlas, denoising)
                        idx_rois4, idx_vls4, _ = load_labels(subjectid, MAIN_PATH, atlas)
                        prf_model = filter_prf_data(prf_params, prf_voxels)
                        generate_combined_plot(subjectid, prf_model, idx_vls4, prf_voxels, rois_list, pdf_pages, OUTPUT_DIR)
                    else:
                        generate_plots(subjectid, MAIN_PATH, atlas=atlas, denoising=denoising, project=project, subject_num=subjectid, pdf_pages=pdf_pages, basic=True, output_dir=OUTPUT_DIR)
                except Exception as e:
                    print(f"Skipping subject {subjectid} due to error: {e}")
        finally:
            if pdf_pages:
                pdf_pages.close()

    elif analysis == 'normality':
        # Load the data from CSV
        csv_path = os.path.join(MAIN_PATH, 'trendline_data', 'trendline_data.csv')
        data = pd.read_csv(csv_path)
        data['Group'] = np.where(data['Subject Number'].isin(glaucoma_subjects), 'Primary Open-Angle Glaucoma',
                                 np.where(data['Subject Number'].isin(controls_subjects), 'Healthy Controls', ''))

        visual_areas = ['V1', 'V2', 'V3']
        groups = {'Primary Open-Angle Glaucoma': 'red', 'Healthy Controls': 'blue'}

        # Calculate normality and save results
        normality_slope(data, MAIN_PATH, visual_areas, groups, atlas)
        normality_data(subjects, MAIN_PATH, atlas, denoising)
    
    # Update the main script section to call the Mann-Whitney U test function
    elif analysis == 'mannwhitney':
        # Load the data from CSV
        csv_path = os.path.join(MAIN_PATH, 'trendline_data', 'trendline_data.csv')
        data = pd.read_csv(csv_path)
        data['Group'] = np.where(data['Subject Number'].isin(glaucoma_subjects), 'Primary Open-Angle Glaucoma',
                                np.where(data['Subject Number'].isin(controls_subjects), 'Healthy Controls', ''))

        visual_areas = ['V1', 'V2', 'V3']
        groups = {'Primary Open-Angle Glaucoma': 'red', 'Healthy Controls': 'blue'}

        # Perform Mann-Whitney U tests and plot trend lines
        perform_mannwhitney_and_plot_trendlines(data, MAIN_PATH, visual_areas, groups, atlas)

looking for ['/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/freesurfer/sub-02/label/lh.benson14_varea-0001.label', '/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/freesurfer/sub-02/label/rh.benson14_varea-0001.label']
looking for ['/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/freesurfer/sub-02/label/lh.benson14_eccen-0001.label', '/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/freesurfer/sub-02/label/rh.benson14_eccen-0001.label']
looking for ['/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/freesurfer/sub-02/label/lh.manualdelin.label', '/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/freesurfer/sub-02/label/rh.manualdelin.label']
looking for ['/Volumes/FedericaCardillo/pre-processing/projects/PROJECT_EGRET-AAA/derivatives/freesurfer/sub-04/label/lh.benson14_varea-0001.label', '/Volumes/Fe